# 🚀 Модуль 10. Урок 1. Работа с LLM API в Python: OpenAI, Anthropic и Google Gemini

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_10_llm_api/lesson_01_calling_llm_apis_python.ipynb)

## 📌 Бизнес-контекст и методология
В 2027 году аналитик использует LLM API как функцию в data-пайплайнах:
1. Архитектура Client-Request: передача сообщений (role: system / user / assistant).
2. Параметры генерации: `temperature` (0.0 для детерминированного анализа данных, 0.7 для креатива), `max_tokens`.
3. Обработка Rate Limits и Exponential Backoff при сетевых ошибках.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import json
import time

# Моделирование унифицированного клиента вызова LLM API с повторными попытками
def mock_llm_api_call(messages, temperature=0.0, max_retries=3):
    """Имитация вызова API с защитой Exponential Backoff."""
    for attempt in range(max_retries):
        try:
            # Симулируем успешный вызов
            system_role = [m['content'] for m in messages if m['role'] == 'system'][0]
            user_msg = [m['content'] for m in messages if m['role'] == 'user'][0]
            response_payload = {
                "id": "chatcmpl-mock-2027",
                "model": "gpt-4o-mini",
                "temperature": temperature,
                "usage": {"prompt_tokens": 45, "completion_tokens": 20, "total_tokens": 65},
                "choices": [{
                    "message": {
                        "role": "assistant",
                        "content": f"Аналитический вывод по запросу '{user_msg}': Зафиксирован рост конверсии на 3.4%."
                    }
                }]
            }
            return response_payload
        except Exception as e:
            wait = 2 ** attempt
            time.sleep(wait)
    raise RuntimeError("API недоступен после всех попыток")

test_messages = [
    {"role": "system", "content": "Ты Staff Data Analyst."},
    {"role": "user", "content": "Проанализируй динамику A/B теста #42"}
]
resp = mock_llm_api_call(test_messages)
print("Ответ API:")
print(resp['choices'][0]['message']['content'])
print(f"Потреблено токенов: {resp['usage']['total_tokens']}")


Ответ API:
Аналитический вывод по запросу 'Проанализируй динамику A/B теста #42': Зафиксирован рост конверсии на 3.4%.
Потреблено токенов: 65


### 🎯 Практическое задание 1: Проверка детерминированности температуры
Для строгих математических расчетов температура обязана быть равна 0.0.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert resp['temperature'] == 0.0, "Для аналитики всегда используйте temperature=0.0"
assert "usage" in resp, "Ответ API должен возвращать метрики токенов"
print("✅ Тест успешно пройден: стандарты работы с API соблюдены!")


✅ Тест успешно пройден: стандарты работы с API соблюдены!


### 💡 Почему temperature=0.0 критична для аналитики?
Ненулевая температура вносит стохастический шум, из-за чего один и тот же отчет о выручке при повторном запуске может выдать разные цифры.